In [ ]:
import os
import cv2
import math
import numpy as np
import pandas as pd
import joblib
from ultralytics import YOLO
from pandas.errors import EmptyDataError

# ============================================================
# 🎛️ SYSTEM CONTROLS & WEIGHTAGE (TWEAK THESE)
# ============================================================
INPUT_VIDEO_PATH = r"/content/Shoot_5.mp4"
OUTPUT_VIDEO_PATH = r"/content/Final_Snatch_Output.mp4"
ML_MODEL_PATH = r"/content/snatch_model.pkl"

# --- 1. ML SENSITIVITY ---
# How sure does the AI need to be before we even bother checking physics? (0.0 to 1.0)
ML_CONFIDENCE_THRESHOLD = 0.50

# --- 2. TEMPORAL SMOOTHING (The "Rule of N") ---
# How many consecutive windows must pass the checks to be a CONFIRMED snatch? (Removes 1-frame glitches)
MIN_CONSECUTIVE_WINDOWS = 3

# --- 3. PHYSICS VETO: DISTANCE & OCCLUSION ---
USE_PHYSICS_VETO = True
PHYSICS_DISTANCE_MAX = 500#120      # VETO if feet are further apart than this (pixels) at the time of contact.
OCCLUSION_DISTANCE_MIN = 15     # VETO if feet distance drops below this (pixels) without object loss (handles tracker ID mergers/glitches).

# --- 4. PHYSICS VETO: HANDSHAKE & SPRINT FILTER ---
MIN_CONTACT_DURATION = 5
MAX_CONTACT_DURATION = 45       # VETO if contact lasts longer than this (frames).
VIOLENT_ESCAPE_SPEED = 150      # The RELATIVE speed (separation speed) must be higher than this.
ESCAPE_SPRINT_SPEED = 180       # The ABSOLUTE MAX speed must be higher than this.
ESCAPE_AVG_SPEED = 50           # ADDED: The AVERAGE speed must also be high to prove it's a real sprint, not a 1-frame tracking glitch.


# ============================================================
# DIRECTORY SETUP
# ============================================================
BASE_OUTPUT_DIR = r"Multi_Modal_SnatchDetection\Dataset\CSV_Extracted"
for d in ["Frame_Entities", "Person_Motion", "Pair_Interaction", "Hand_Interaction", "Object_Interaction", "Features"]:
    os.makedirs(os.path.join(BASE_OUTPUT_DIR, d), exist_ok=True)

VIDEO_ID = os.path.splitext(os.path.basename(INPUT_VIDEO_PATH))[0]
F_ENT = os.path.join(BASE_OUTPUT_DIR, "Frame_Entities", f"{VIDEO_ID}_frame_entities.csv")
F_MOT = os.path.join(BASE_OUTPUT_DIR, "Person_Motion", f"{VIDEO_ID}_person_motion.csv")
F_PAIR = os.path.join(BASE_OUTPUT_DIR, "Pair_Interaction", f"{VIDEO_ID}_person_pair_interaction.csv")
F_HAND = os.path.join(BASE_OUTPUT_DIR, "Hand_Interaction", f"{VIDEO_ID}_hand_interaction.csv")
F_OBJ = os.path.join(BASE_OUTPUT_DIR, "Object_Interaction", f"{VIDEO_ID}_object_interaction.csv")
F_FEAT = os.path.join(BASE_OUTPUT_DIR, "Features", f"{VIDEO_ID}_features.csv")

print(f"🚀 INITIALIZING PIPELINE FOR: {VIDEO_ID}")

# ============================================================
# PHASE 1: FULL FEATURE EXTRACTION
# ============================================================
print("\n--- PHASE 1: EXTRACTING FEATURES ---")
model = YOLO("yolov8n.pt")
cap = cv2.VideoCapture(INPUT_VIDEO_PATH)
fps = cap.get(cv2.CAP_PROP_FPS)
if fps <= 1: fps = 30.0
cap.release()

def euclidean(p1, p2): return math.sqrt((p1[0] - p2[0])**2 + (p1[1] - p2[1])**2)

# --- TABLE 1: ENTITIES ---
print("Tracking Entities...")
frame_entities, object_tracker, next_object_id, frame_id = [], {}, 0, 0
results_generator = model.track(source=INPUT_VIDEO_PATH, tracker="botsort.yaml", persist=True, conf=0.35, stream=True, verbose=False)
for result in results_generator:
    frame_id += 1
    if result.boxes is None: continue
    boxes, class_ids, confs = result.boxes.xyxy.cpu().numpy(), result.boxes.cls.cpu().numpy(), result.boxes.conf.cpu().numpy()
    track_ids = result.boxes.id.cpu().numpy() if result.boxes.id is not None else [None] * len(boxes)
    for i, box in enumerate(boxes):
        cls_id, conf = int(class_ids[i]), float(confs[i])
        x1, y1, x2, y2 = map(int, box)
        cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
        if cls_id in {0, 1, 3}:
            if track_ids[i] is None: continue
            frame_entities.append({"frame_id": frame_id, "timestamp_sec": frame_id/fps, "entity_type": "person", "entity_id": int(track_ids[i]), "class_name": "person", "x1": x1, "y1": y1, "x2": x2, "y2": y2, "bbox_center_x": cx, "bbox_center_y": cy, "bbox_height": y2-y1, "bbox_width": x2-x1})
        elif cls_id in {24, 26, 28, 67}:
            matched_id, min_dist = None, float("inf")
            for oid, (ox, oy, last_f) in object_tracker.items():
                if frame_id - last_f > 5: continue
                dist = euclidean((cx, cy), (ox, oy))
                if dist < 80 and dist < min_dist: min_dist, matched_id = dist, oid
            eid = matched_id if matched_id is not None else next_object_id
            if matched_id is None: next_object_id += 1
            object_tracker[eid] = (cx, cy, frame_id)
            frame_entities.append({"frame_id": frame_id, "timestamp_sec": frame_id/fps, "entity_type": "object", "entity_id": eid, "class_name": "object", "x1": x1, "y1": y1, "x2": x2, "y2": y2, "bbox_center_x": cx, "bbox_center_y": cy, "bbox_height": y2-y1, "bbox_width": x2-x1})
df_entities = pd.DataFrame(frame_entities)

# --- TABLE 2: MOTION ---
print("Extracting Motion...")
df_persons = df_entities[df_entities["entity_type"] == "person"].copy()
person_motion_rows = []
if not df_persons.empty:
    df_persons.sort_values(by=["entity_id", "frame_id"], inplace=True)
    df_persons['smooth_cx'] = df_persons.groupby('entity_id')['bbox_center_x'].transform(lambda x: x.ewm(alpha=0.6, adjust=False).mean())
    df_persons['smooth_cy'] = df_persons.groupby('entity_id')['bbox_center_y'].transform(lambda x: x.ewm(alpha=0.6, adjust=False).mean())
    prev_state = {}
    for _, row in df_persons.iterrows():
        pid, fid, cx, cy = row["entity_id"], row["frame_id"], row["smooth_cx"], row["smooth_cy"]
        if pid not in prev_state:
            prev_state[pid] = {"cx": cx, "cy": cy, "speed": 0.0, "time": row["timestamp_sec"]}
            vx = vy = speed = accel = 0.0
        else:
            prev = prev_state[pid]
            dt = max(row["timestamp_sec"] - prev["time"], 1/fps)
            vx, vy = (cx - prev["cx"]) / dt, (cy - prev["cy"]) / dt
            speed = math.sqrt(vx**2 + vy**2)
            accel = (speed - prev["speed"]) / dt
            prev_state[pid] = {"cx": cx, "cy": cy, "speed": speed, "time": row["timestamp_sec"]}
        person_motion_rows.append({"frame_id": fid, "person_id": pid, "center_x": cx, "center_y": cy, "velocity_x": vx, "velocity_y": vy, "speed": speed, "acceleration": accel})
df_motion = pd.DataFrame(person_motion_rows)

# --- TABLE 3: PAIRS ---
print("Extracting Pairs (Using Footpoint Distance)...")
pair_rows = []
if not df_motion.empty:
    df_person_boxes = df_entities[df_entities["entity_type"] == "person"].set_index(["frame_id", "entity_id"])
    grouped = df_motion.groupby("frame_id")
    for fid, frame_df in grouped:
        persons = frame_df.to_dict("records")
        for i in range(len(persons)):
            for j in range(i+1, len(persons)):
                p1, p2 = persons[i], persons[j]
                id1, id2 = int(p1["person_id"]), int(p2["person_id"])

                try:
                    b1, b2 = df_person_boxes.loc[(fid, id1)], df_person_boxes.loc[(fid, id2)]
                    f1_x, f1_y = b1["bbox_center_x"], b1["y2"]
                    f2_x, f2_y = b2["bbox_center_x"], b2["y2"]
                except KeyError:
                    f1_x, f1_y = p1["center_x"], p1["center_y"]
                    f2_x, f2_y = p2["center_x"], p2["center_y"]

                dx, dy = f1_x - f2_x, f1_y - f2_y
                dist = math.sqrt(dx**2 + dy**2) + 1e-6

                rvx, rvy = p1["velocity_x"] - p2["velocity_x"], p1["velocity_y"] - p2["velocity_y"]
                rel_speed = math.sqrt(rvx**2 + rvy**2)
                approach_vel = (rvx * (dx/dist)) + (rvy * (dy/dist))
                pair_rows.append({"frame_id": fid, "person_i_id": id1, "person_j_id": id2, "distance_ground": dist, "approach_velocity": approach_vel, "relative_speed": rel_speed})
df_pair = pd.DataFrame(pair_rows)

# --- TABLE 4: HANDS ---
print("Extracting Hands...")
hand_rows = []
if not df_persons.empty:
    grouped = df_persons.groupby("frame_id")
    mem = {}
    for fid, frame_df in grouped:
        persons = frame_df.to_dict("records")
        for p_i in persons:
            for p_j in persons:
                if p_i["entity_id"] == p_j["entity_id"]: continue
                for hx_f in [0.25, 0.75]:
                    hx = p_i["x1"] + hx_f * p_i["bbox_width"]
                    hy = p_i["y1"] + 0.35 * p_i["bbox_height"]
                    dist = math.sqrt((hx - p_j["bbox_center_x"])**2 + (hy - p_j["bbox_center_y"])**2)
                    if dist > 80: continue
                    key = (p_i["entity_id"], p_j["entity_id"], hx_f)
                    count = mem[key]["count"] + 1 if key in mem and fid - mem[key]["last"] <= 2 else 1
                    mem[key] = {"count": count, "last": fid}
                    hand_rows.append({"frame_id": fid, "person_i_id": p_i["entity_id"], "contact_duration_frames": count})
df_hand = pd.DataFrame(hand_rows)

# --- TABLE 5: OBJECTS ---
print("Extracting Objects...")
object_rows = []
df_objects = df_entities[df_entities["entity_type"] == "object"]
if not df_objects.empty and not df_motion.empty:
    obj_grp = df_objects.groupby("frame_id")
    per_grp = df_motion.groupby("frame_id")
    obj_mem = {}
    for fid in sorted(df_entities["frame_id"].unique()):
        curr_objs = obj_grp.get_group(fid) if fid in obj_grp.groups else pd.DataFrame()
        curr_pers = per_grp.get_group(fid) if fid in per_grp.groups else pd.DataFrame()
        curr_ids = set()
        for _, obj in curr_objs.iterrows():
            oid = obj["entity_id"]
            curr_ids.add(oid)
            cx, cy = obj["bbox_center_x"], obj["bbox_center_y"]
            o_id, o_dist, o_vx, o_vy = -1, float("inf"), 0, 0
            for _, p in curr_pers.iterrows():
                dist = math.sqrt((cx - p["center_x"])**2 + (cy - p["center_y"])**2)
                if dist < o_dist and dist < 150: o_dist, o_id, o_vx, o_vy = dist, p["person_id"], p["velocity_x"], p["velocity_y"]

            ovx = ovy = 0
            if oid in obj_mem and 0 < (fid - obj_mem[oid]["last_f"]) <= 5:
                dt = fid - obj_mem[oid]["last_f"]
                ovx, ovy = (cx - obj_mem[oid]["cx"])/dt, (cy - obj_mem[oid]["cy"])/dt
            obj_mem[oid] = {"last_f": fid, "cx": cx, "cy": cy, "status": "tracked"}

            match = 0.0
            if o_id != -1:
                mo, mp = math.sqrt(ovx**2 + ovy**2), math.sqrt(o_vx**2 + o_vy**2)
                if mo > 1e-6 and mp > 1e-6: match = (ovx*o_vx + ovy*o_vy) / (mo*mp)

            object_rows.append({"frame_id": fid, "object_id": oid, "disappearance_flag": 0, "attacker_velocity_match": match})

        for oid, m in obj_mem.items():
            if oid not in curr_ids and m["status"] != "lost" and fid - m["last_f"] > 3:
                object_rows.append({"frame_id": fid, "object_id": oid, "disappearance_flag": 1, "attacker_velocity_match": 0.0})
                m["status"] = "lost"
df_object = pd.DataFrame(object_rows)

# --- TABLE 6: WINDOW FEATURES ---
print("Aggregating ML Features...")
df_pair = df_pair if not df_pair.empty else pd.DataFrame(columns=["frame_id", "distance_ground", "approach_velocity", "relative_speed"])
df_hand = df_hand if not df_hand.empty else pd.DataFrame(columns=["frame_id", "contact_duration_frames", "person_i_id"])
df_object = df_object if not df_object.empty else pd.DataFrame(columns=["frame_id", "disappearance_flag", "attacker_velocity_match"])

window_features = []
max_frame = frame_id
for start_frame in range(1, int(max_frame), 15):
    end_frame = start_frame + 30
    w_mot = df_motion[(df_motion['frame_id'] >= start_frame) & (df_motion['frame_id'] <= end_frame)]
    w_par = df_pair[(df_pair['frame_id'] >= start_frame) & (df_pair['frame_id'] <= end_frame)]
    w_hnd = df_hand[(df_hand['frame_id'] >= start_frame) & (df_hand['frame_id'] <= end_frame)]
    w_obj = df_object[(df_object['frame_id'] >= start_frame) & (df_object['frame_id'] <= end_frame)]
    if w_mot.empty: continue

    window_features.append({
        "window_id": f"w_{start_frame}_{end_frame}", "start_frame": start_frame, "end_frame": end_frame,
        "avg_speed": round(w_mot['speed'].mean(), 2), "max_speed": min(round(w_mot['speed'].max(), 2), 300),
        "max_acceleration": min(round(w_mot['acceleration'].max(), 2), 5000),
        "min_distance_ground": round(w_par['distance_ground'].min(), 2) if not w_par.empty else 999.0,
        "avg_approach_velocity": round(w_par['approach_velocity'].mean(), 2) if not w_par.empty else 0.0,
        "max_relative_speed": round(w_par['relative_speed'].max(), 2) if not w_par.empty else 0.0,
        "max_contact_duration": int(w_hnd['contact_duration_frames'].max()) if not w_hnd.empty else 0,
        "contact_events_count": int(w_hnd['person_i_id'].nunique()) if not w_hnd.empty else 0,
        "object_lost_flag": 1 if (not w_obj.empty and (w_obj['disappearance_flag'] == 1).any()) else 0,
        "max_velocity_match": round(w_obj['attacker_velocity_match'].max(), 2) if not w_obj.empty else 0.0
    })
df_features = pd.DataFrame(window_features)
print("✅ Features Assembled.")

# ============================================================
# PHASE 2: ML INFERENCE & DETAILED LOGGING (DEBUG MODE)
# ============================================================
print("\n--- PHASE 2: ML INFERENCE ---")
try:
    ml_model = joblib.load(ML_MODEL_PATH)
except FileNotFoundError:
    raise FileNotFoundError(f"❌ Model not found at {ML_MODEL_PATH}")

df_features["speed_change"] = df_features["max_speed"] - df_features["avg_speed"]
df_features["interaction_intensity"] = df_features["max_relative_speed"] * df_features["max_contact_duration"]

X_infer = df_features.drop(columns=["window_id", "start_frame", "end_frame", "label", "avg_speed"], errors='ignore')
X_infer = X_infer.apply(pd.to_numeric, errors='coerce').fillna(0)
for col in ml_model.get_booster().feature_names:
    if col not in X_infer.columns: X_infer[col] = 0.0
X_infer = X_infer[ml_model.get_booster().feature_names]

probs = ml_model.predict_proba(X_infer)[:, 1]
frame_data = {}

print("\n🔍 WINDOW EVALUATION LOG (ADVANCED DEBUG):")
print(f"{'WINDOW':<12} | {'PROB':<5} | {'DIST':<4} | {'MAXS':<4} | {'AVGS':<4} | {'REL':<4} | {'DUR':<3} | {'OBJ':<3} | {'RESULT'}")
print("-" * 85)

consecutive_snatch_count = 0

for i, row in df_features.iterrows():
    prob = probs[i]
    window_name = row["window_id"]
    min_dist = row["min_distance_ground"]
    max_spd = row["max_speed"]
    avg_spd = row["avg_speed"]  # Fetch the average speed
    rel_spd = row["max_relative_speed"]
    contact_dur = row["max_contact_duration"]

    # 1. Base Variables
    has_loss = row["object_lost_flag"] == 1
    # Veto contact that is too fast (a pass-by) or too slow (a handshake)
    is_quick_contact = (MIN_CONTACT_DURATION <= contact_dur <= MAX_CONTACT_DURATION)

    # 🚨 THE FIX: Require both MAX speed and AVG speed to be high to ignore glitches
    is_sprinting = (max_spd >= ESCAPE_SPRINT_SPEED) and (avg_spd >= ESCAPE_AVG_SPEED)
    is_violent_separation = rel_spd >= VIOLENT_ESCAPE_SPEED

    # 2. Strict Veto Logic
    if (min_dist < OCCLUSION_DISTANCE_MIN) and not has_loss:
        veto_passed = False
        veto_reason = "VETO (0px/Glitch)"
    elif (min_dist > PHYSICS_DISTANCE_MAX) and not has_loss:
        veto_passed = False
        veto_reason = "VETO (Too Far)"
    elif has_loss:
        veto_passed = True
        veto_reason = ""
    elif is_quick_contact and is_sprinting and is_violent_separation:
        veto_passed = True
        veto_reason = ""
    else:
        veto_passed = False if USE_PHYSICS_VETO else True
        if contact_dur > 0 and contact_dur < MIN_CONTACT_DURATION:
            veto_reason = "VETO (Pass-By/Too Fast)"
        else:
            veto_reason = "VETO (Slow/Handshake)"

    is_suspicious = (prob >= ML_CONFIDENCE_THRESHOLD) and veto_passed

    # 3. Temporal Smoothing (The Rule of N)
    if is_suspicious:
        consecutive_snatch_count += 1
    else:
        consecutive_snatch_count = 0

    is_confirmed_snatch = consecutive_snatch_count >= MIN_CONSECUTIVE_WINDOWS

    # Logging text
    if is_confirmed_snatch:
        res_text = "🚨 CONFIRMED SNATCH"
    elif is_suspicious:
        res_text = f"⚠️ BUILDING ({consecutive_snatch_count}/{MIN_CONSECUTIVE_WINDOWS})"
    else:
        res_text = veto_reason if prob >= ML_CONFIDENCE_THRESHOLD else "NORMAL"

    # Print the raw data including AVGS
    print(f"{window_name:<12} | {prob*100:>4.1f}% | {min_dist:<4.0f} | {max_spd:<4.0f} | {avg_spd:<4.0f} | {rel_spd:<4.0f} | {contact_dur:<3.0f} | {str(has_loss)[0]:<3} | {res_text}")

    for f in range(int(row["start_frame"]), int(row["end_frame"]) + 1):
        if f not in frame_data or prob > frame_data[f]['prob']:
            frame_data[f] = {
                'prob': prob,
                'is_snatch': is_confirmed_snatch,
                'veto': veto_passed,
                'dist': min_dist,
                'obj_lost': has_loss,
                'res_text': res_text
            }


# ============================================================
# PHASE 3: VISUALIZATION RENDERER
# ============================================================
print("\n--- PHASE 3: RENDERING OUTPUT VIDEO ---")
cap = cv2.VideoCapture(INPUT_VIDEO_PATH)
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(cap.get(cv2.CAP_PROP_FPS))
if fps <= 1: fps = 30

out = cv2.VideoWriter(OUTPUT_VIDEO_PATH, cv2.VideoWriter_fourcc(*'mp4v'), fps, (w, h))

def index_by_frame(df): return {k: v for k, v in df.groupby("frame_id")} if not df.empty else {}
entities_map = index_by_frame(df_entities)
motion_map = index_by_frame(df_motion)
FONT = cv2.FONT_HERSHEY_SIMPLEX

def draw_text(img, text, pos, color, scale=0.5, thickness=2):
    cv2.putText(img, text, pos, FONT, scale, (0,0,0), thickness + 2)
    cv2.putText(img, text, pos, FONT, scale, color, thickness)

frame_id = 0
while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break
    frame_id += 1

    ents = entities_map.get(frame_id, pd.DataFrame())
    mots = motion_map.get(frame_id, pd.DataFrame())

    for _, e in ents.iterrows():
        x1, y1, x2, y2 = int(e.x1), int(e.y1), int(e.x2), int(e.y2)
        if e.entity_type == "person":
            cv2.rectangle(frame, (x1,y1), (x2,y2), (0, 255, 0), 2)
            m = mots[mots.person_id == e.entity_id]
            if not m.empty and m.iloc[0].speed > 5:
                vx, vy = m.iloc[0].velocity_x, m.iloc[0].velocity_y
                cx, cy = int(e.bbox_center_x), int(e.bbox_center_y)
                cv2.arrowedLine(frame, (cx, cy), (int(cx + vx*2), int(cy + vy*2)), (255, 255, 0), 2, tipLength=0.3)
        else:
            cv2.rectangle(frame, (x1,y1), (x2,y2), (0, 165, 255), 2)

    if frame_id in frame_data:
        dat = frame_data[frame_id]
        ai_prob = dat['prob'] * 100

        hud_color = (0, 0, 255) if dat['is_snatch'] else (0, 255, 0)
        draw_text(frame, f"AI Probability: {ai_prob:.1f}%", (20, 40), hud_color, 0.7)
        draw_text(frame, f"Foot Distance: {dat['dist']:.1f}px", (20, 70), (255, 255, 255), 0.6)
        draw_text(frame, f"Object Lost: {'Yes' if dat['obj_lost'] else 'No'}", (20, 100), (255, 255, 255), 0.6)
        draw_text(frame, f"Status: {dat['res_text']}", (20, 130), (255, 255, 255), 0.6)

        if dat['is_snatch']:
            cv2.rectangle(frame, (0, 0), (w, h), (0, 0, 255), 8)
            draw_text(frame, "🚨 CONFIRMED SNATCH!", (w//2 - 250, 100), (0, 0, 255), 1.5, 3)

    out.write(frame)

cap.release()
out.release()
try:
    cv2.destroyAllWindows()
except cv2.error:
    pass
print(f"✅ ALL DONE! Output saved to: {OUTPUT_VIDEO_PATH}")

🚀 INITIALIZING PIPELINE FOR: Shoot_5

--- PHASE 1: EXTRACTING FEATURES ---
Tracking Entities...
Extracting Motion...
Extracting Pairs (Using Footpoint Distance)...
Extracting Hands...
Extracting Objects...
Aggregating ML Features...
✅ Features Assembled.

--- PHASE 2: ML INFERENCE ---

🔍 WINDOW EVALUATION LOG (ADVANCED DEBUG):
WINDOW       | PROB  | DIST | MAXS | AVGS | REL  | DUR | OBJ | RESULT
-------------------------------------------------------------------------------------
w_1_31       |  0.2% | 999  | 54   | 12   | 0    | 0   | F   | NORMAL
w_16_46      |  0.3% | 999  | 19   | 8    | 0    | 0   | F   | NORMAL
w_31_61      |  7.7% | 999  | 29   | 9    | 0    | 0   | F   | NORMAL
w_46_76      |  0.2% | 999  | 71   | 23   | 0    | 0   | F   | NORMAL
w_61_91      |  0.2% | 999  | 71   | 32   | 0    | 0   | F   | NORMAL
w_76_106     |  0.2% | 999  | 65   | 30   | 0    | 0   | F   | NORMAL
w_91_121     |  0.2% | 999  | 93   | 44   | 0    | 0   | F   | NORMAL
w_106_136    |  0.2% | 99

In [ ]:
%pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 4.4 MB/s eta 0:00:00
